In [52]:
import sys
sys.path.append("/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR")

# Experimentation with PyOD Models

This notebook evaluates PyOD models on two UCI datasets that are more suitable for anomaly detection: `shuttle` and `arrhythmia`. The goal is to load the datasets from the RADAR static dataset module, reframe them as anomaly-detection benchmarks, and compare several PyOD models using label-based and score-based metrics.

In [59]:
# Import Required Libraries
import importlib

import numpy as np
import pandas as pd

from RADAR.static_data.algorithms import pyod
import RADAR.metrics_module as metrics_module

metrics_module = importlib.reload(metrics_module)

In [54]:
import RADAR.static_data.anomaly_dataset_utils as anomaly_dataset_utils

anomaly_dataset_utils = importlib.reload(anomaly_dataset_utils)

uci_dataset_configs = {
    "shuttle": anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
        dataset_name="shuttle",
        normal_label=1,
        target_test_contamination=0.1,
        max_train_normals=8000,
        max_test_size=5000,
    ),
    "arrhythmia": anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
        dataset_name="arrhythmia",
        normal_label=1,
        target_test_contamination=0.1,
    ),
}

In [ ]:
uci_summary_rows = []
for dataset_name, config in uci_dataset_configs.items():
    uci_summary_rows.append(
        {
            "dataset": dataset_name,
            "samples": config["n_samples"],
            "features": config["n_features"],
            "original_anomaly_ratio": round(config["original_positive_ratio"], 4),
            "benchmark_test_contamination": round(
                config["benchmark_test_positive_ratio"], 4
            ),
            "train_normals_used": config["train_normals"],
            "test_normals": config["test_normals"],
            "test_anomalies": config["test_anomalies"],
        }
    )

uci_summary_df = pd.DataFrame(uci_summary_rows)
display(uci_summary_df)

### PyOD Experiment on Shuttle and Arrhythmia



In [ ]:
uci_pyod_models = [
    {"algorithm_": "cblof"},
    {"algorithm_": "iforest", "random_state": 42},
    {"algorithm_": "knn", "n_neighbors": 5},
    {"algorithm_": "hbos"},
    {"algorithm_": "ocsvm"},
    {"algorithm_": "lof", "n_neighbors": 5},
]

uci_results = []

for dataset_name, config in uci_dataset_configs.items():
    print(f"\nDataset: {dataset_name}")
    print(
        f"Training with normal-only samples: {config['train_normals']} | "
        f"Benchmark contamination: {config['benchmark_test_positive_ratio']:.3f}"
    )

    for model_params in uci_pyod_models:
        model_kwargs = {
            **model_params,
            "contamination": config["benchmark_test_positive_ratio"],
        }

        model = pyod.PyodAnomalyDetection(**model_kwargs)
        model.fit(config["X_train"])
        predictions = np.asarray(model.predict(config["X_test"])).astype(int).ravel()
        scores = np.asarray(model.decision_function(config["X_test"])).ravel()

        accuracy = metrics_module.metric_accuracy(config["y_test"], predictions) / 100
        precision = metrics_module.metric_precision(config["y_test"], predictions)
        recall = metrics_module.metric_recall(config["y_test"], predictions)
        f1 = metrics_module.metric_F1score(config["y_test"], predictions)

        finite_scores = np.isfinite(scores)
        if finite_scores.all():
            roc_auc = metrics_module.metric_AUC_ROC_scores(config["y_test"], scores)
            pr_auc = metrics_module.metric_PR_AUC(config["y_test"], scores)
            score_note = ""
        else:
            roc_auc = np.nan
            pr_auc = np.nan
            score_note = " | score metrics skipped (NaN decision scores)"

        print(f"\nModel: {model_params['algorithm_']}{score_note}")
        metrics_module.print_metrics(["Accuracy", "Precision", "Recall", "F1"], config["y_test"], predictions)
        if finite_scores.all():
            print(f"ROC AUC (scores): {roc_auc:.3f}")
            print(f"PR AUC (scores): {pr_auc:.3f}")

        uci_results.append(
            {
                "dataset": dataset_name,
                "algorithm": model_params["algorithm_"],
                "contamination": round(config["benchmark_test_positive_ratio"], 4),
                "accuracy": round(accuracy, 4),
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "f1": round(f1, 4),
                "roc_auc_scores": round(float(roc_auc), 4) if np.isfinite(roc_auc) else np.nan,
                "pr_auc_scores": round(float(pr_auc), 4) if np.isfinite(pr_auc) else np.nan,
            }
        )

uci_results_df = pd.DataFrame(uci_results).sort_values(
    ["dataset", "pr_auc_scores", "roc_auc_scores"],
    ascending=[True, False, False],
    na_position="last",
).reset_index(drop=True)

display(uci_results_df)

### Timing Comparison: RADAR vs Direct PyOD

### How to Interpret the Timing Table

- `dataset`: dataset on which the comparison was run (`shuttle` or `arrhythmia`).
- `algorithm`: PyOD model being evaluated.
- `platform_time_s`: total execution time in seconds using the RADAR wrapper.
- `base_time_s`: total execution time in seconds using the direct PyOD class.
- `speedup_base_over_platform`: ratio `base_time_s / platform_time_s`. 
- `platform_roc_auc_scores`: score-based ROC-AUC obtained with the RADAR implementation.
- `base_roc_auc_scores`: score-based ROC-AUC obtained with the direct PyOD implementation.
- `roc_auc_diff`: difference `platform_roc_auc_scores - base_roc_auc_scores`. It shows whether the RADAR path preserves or changes ranking quality relative to direct PyOD.

### When Is One Better Than the Other?

- For **runtime**, RADAR is faster when `speedup_base_over_platform > 1` because the direct baseline took more time than the platform.
- If `speedup_base_over_platform < 1`, the direct PyOD implementation is faster.
- If `speedup_base_over_platform ≈ 1`, both approaches have very similar runtime.
- For **quality**, higher `platform_roc_auc_scores` or `base_roc_auc_scores` is better because ROC-AUC closer to `1.0` means better separation between normal samples and anomalies.
- If `roc_auc_diff > 0`, RADAR gives better score ranking quality than the direct baseline for that model and dataset.
- If `roc_auc_diff < 0`, the direct PyOD version gives better score ranking quality.
- Ideally, the preferred case is: `speedup_base_over_platform > 1` and `roc_auc_diff >= 0`, meaning RADAR is faster while keeping equal or better ROC-AUC.
- If one approach is faster but has lower ROC-AUC, then it is a trade-off between efficiency and detection quality.

In [ ]:
import time
from statistics import mean
from tqdm import tqdm
 
direct_pyod_algorithms = {
    "cblof": CBLOF,
    "iforest": IForest,
    "knn": KNN,
    "hbos": HBOS,
    "ocsvm": OCSVM,
    "lof": LOF,
}
 
uci_timing_results = []
 
# Define the number of repetitions
num_repetitions = 30
 
for dataset_name, config in uci_dataset_configs.items():
    for model_params in uci_pyod_models:
        algorithm_name = model_params["algorithm_"]
        shared_kwargs = {
            key: value
            for key, value in model_params.items()
            if key != "algorithm_"
        }
 
        platform_model = pyod.PyodAnomalyDetection(
            algorithm_=algorithm_name,
            contamination=config["benchmark_test_positive_ratio"],
            **shared_kwargs,
        )
 
        # Perform multiple repetitions for platform execution
        platform_execution_times = []
        for _ in tqdm(range(num_repetitions), desc=f"Platform Execution Timing ({algorithm_name})"):
            start_time = time.time()
            platform_model.fit(config["X_train"])
            platform_execution_times.append(time.time() - start_time)
 
        direct_model_cls = direct_pyod_algorithms[algorithm_name]
        direct_model = direct_model_cls(
            contamination=config["benchmark_test_positive_ratio"],
            **shared_kwargs,
        )
 
        # Perform multiple repetitions for direct execution
        direct_execution_times = []
        for _ in tqdm(range(num_repetitions), desc=f"Direct Execution Timing ({algorithm_name})"):
            start_time = time.time()
            direct_model.fit(config["X_train"])
            direct_execution_times.append(time.time() - start_time)
 
        # Calculate average execution times
        average_platform_time = mean(platform_execution_times)
        average_direct_time = mean(direct_execution_times)
 
        platform_predictions = np.asarray(
            platform_model.predict(config["X_test"])
        ).astype(int).ravel()
        platform_scores = np.asarray(
            platform_model.decision_function(config["X_test"])
        ).ravel()
        platform_roc_auc = (
            metrics_module.metric_AUC_ROC_scores(config["y_test"], platform_scores)
            if np.isfinite(platform_scores).all()
            else np.nan
        )
 
        direct_predictions = np.asarray(
            direct_model.predict(config["X_test"])
        ).astype(int).ravel()
        direct_scores = np.asarray(
            direct_model.decision_function(config["X_test"])
        ).ravel()
        direct_roc_auc = (
            metrics_module.metric_AUC_ROC_scores(config["y_test"], direct_scores)
            if np.isfinite(direct_scores).all()
            else np.nan
        )
 
        # Calculate overhead
        overhead = average_platform_time - average_direct_time
 
        uci_timing_results.append(
            {
                "dataset": dataset_name,
                "algorithm": algorithm_name,
                "average_platform_time_s": round(average_platform_time, 4),
                "average_base_time_s": round(average_direct_time, 4),
                "overhead_s": round(overhead, 4),
                "speedup_base_over_platform": round(
                    average_direct_time / average_platform_time, 4
                ) if average_platform_time > 0 else np.nan,
                "platform_roc_auc_scores": round(float(platform_roc_auc), 4) if np.isfinite(platform_roc_auc) else np.nan,
                "base_roc_auc_scores": round(float(direct_roc_auc), 4) if np.isfinite(direct_roc_auc) else np.nan,
                "roc_auc_diff": round(float(platform_roc_auc - direct_roc_auc), 4)
                if np.isfinite(platform_roc_auc) and np.isfinite(direct_roc_auc)
                else np.nan,
            }
        )
 
uci_timing_df = pd.DataFrame(uci_timing_results).sort_values(
    ["dataset", "speedup_base_over_platform"],
    ascending=[True, False],
).reset_index(drop=True)
 
display(uci_timing_df)

In [ ]:
from pathlib import Path

results_dir = Path("/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results")
results_dir.mkdir(parents=True, exist_ok=True)

results_main_path = results_dir / "uci_pyod_results.csv"
results_timing_path = results_dir / "uci_pyod_timing_results.csv"

uci_results_df.to_csv(results_main_path, index=False)
uci_timing_df.to_csv(results_timing_path, index=False)

print(f"Saved main results to: {results_main_path}")
print(f"Saved timing results to: {results_timing_path}")